In [114]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
from tqdm import tqdm
from sklearn.feature_extraction.text import CountVectorizer
import os

from thefuzz import process
from thefuzz import fuzz

In [115]:
shapefiles_path = "../../data/shapefiles/"
data_folder = "../../data/"

ara_data_folder = "../../data/newsapi/downloads/ara/"
eng_data_folder = "../../data/newsapi/downloads/eng/"

# 1. Associating keywords and locations with articles

## 1.1 Loading Shapefiles and Keywords

In [116]:
wb_shp_path = shapefiles_path + "WB_clean/WB_clean.shp"
wb_shp = gpd.read_file(wb_shp_path)

In [117]:
keyword_df = pd.read_csv(data_folder + "keywords_dataframe.csv")

In [118]:
# Define the order of columns for later use
keyword_columns = keyword_df.loc[~keyword_df["keyword_eng"].isna(), "column_name"].unique()
keyword_columns = sorted(keyword_columns)

location_columns = wb_shp["ID"].unique()
location_columns = sorted(location_columns)

column_order = keyword_columns + location_columns

## 1.2 Processing English news articles

In [119]:
# Loading the raw downloaded data
eng = pd.read_csv(eng_data_folder + "Mashreq_2024-06-23_2024-07-24_daily_article_eng.csv") 
eng = eng.drop(columns=['userHasPermissions'])

In [120]:
# Adding the body length column
eng["body_len"] = eng["body"].apply(lambda x: len(x))

eng["body_len_str"] = eng["body"].apply(lambda x: len(x.split(" ")))

# Converting the body to lowercase
eng["body"] = eng["body"].apply(lambda x: x.lower())

### 1.2.1 English Keywords

In [121]:
keyword_df_eng = keyword_df.loc[~keyword_df["keyword_eng"].isna(), ["keyword_eng", "column_name"]].copy()

keyword_df_eng.drop_duplicates(inplace=True)
keyword_df_eng.reset_index(drop=True, inplace=True)

In [122]:
keyword_df_eng.sort_values("column_name", inplace=True)
keyword_df_eng.reset_index(drop=True, inplace=True)

In [123]:
category_keywords = keyword_df_eng["keyword_eng"].tolist()
category_keywords_column_names = keyword_df_eng["column_name"].tolist()

In [124]:
# The CountVectorizer requires as input the numbers of ngrams to consider, so we need to find the maximum number of ngrams required
ngrams_upper_bound_keywords = np.max([len(word.split(" ")) for word in category_keywords])
print(f"The upper bound for the n-grams is {ngrams_upper_bound_keywords}")

The upper bound for the n-grams is 4


### 1.2.2 Location Names

Count Vectorizer treats words connected with a dash as two distinct words.

Using as vocabulary ['basrah'] and applying it to ['al basrah', 'al-basrah, test', 'basrah, test', 'albasrah'] results in array([[1], [1], [1], [0]]).

In [125]:
def create_multiple_representation_for_locations(locations, locations_column_names):
    """
    Creates multiple representations for the given locations and locations_column_names. 
    The multiple representations include the original locations and locations_column_names, 
    in cases where the location name contains dashes, a dash free version is added to the locations list.
    In cases with Arabic articles in the middle of the name, a version with the article and dash is added to the locations list.
    For all these cases, a corresponding column name is added to the locations_column_names list.  
    
    Args:
        locations (list): A list of locations.
        locations_column_names (list): A list of column names corresponding to the locations.

    Returns:
        tuple: A tuple containing the cleaned locations and cleaned locations_column_names.

    Raises:
        AssertionError: If the number of elements in the cleaned lists is not the same.
    """
    locations_clean = locations.copy()
    locations_column_names_clean = locations_column_names.copy()

    for idx, location_column in enumerate(locations_column_names):
        
        # Dash
        if "-" in location_column:
            print(f"dash: {locations[idx]}")
            location_column_no_dash = location_column + "_no_dash"
            location_no_dash = locations[idx].replace("-", " ")
            locations_column_names_clean.append(location_column_no_dash)
            locations_clean.append(location_no_dash)
            
        # Article in the middle
        if " al " in locations[idx]:
            print(f"al article: {locations[idx]}")
            location_column_al_dash = location_column + "_al_dash"
            location_al_dash = locations[idx].replace(" al ", " al-")
            locations_column_names_clean.append(location_column_al_dash)
            locations_clean.append(location_al_dash)
        
        elif " as " in locations[idx]:
            print(f"as article: {locations[idx]}")
            location_column_as_dash = location_column + "_as_dash"
            location_as_dash = locations[idx].replace(" as ", " as-")
            locations_column_names_clean.append(location_column_as_dash)
            locations_clean.append(location_as_dash)
            
        elif " ar " in locations[idx]:
            print(f"ar article: {locations[idx]}")
            location_column_ar_dash = location_column + "_ar_dash"
            location_ar_dash = locations[idx].replace(" ar ", " ar-")
            locations_column_names_clean.append(location_column_ar_dash)
            locations_clean.append(location_ar_dash)
            
        elif " el " in locations[idx]:
            print(f"el article: {locations[idx]}")
            location_column_el_dash = location_column + "_el_dash"
            location_el_dash = locations[idx].replace(" el ", " el-")
            locations_column_names_clean.append(location_column_el_dash)
            locations_clean.append(location_el_dash)
            
        elif " ath " in locations[idx]:
            print(f"ath article: {locations[idx]}")
            location_column_ath_dash = location_column + "_ath_dash"
            location_ath_dash = locations[idx].replace(" ath ", " ath-")
            locations_column_names_clean.append(location_column_ath_dash)
            locations_clean.append(location_ath_dash) 
        
        elif " az " in locations[idx]:
            print(f"az article: {locations[idx]}")
            location_column_az_dash = location_column + "_az_dash"
            location_az_dash = locations[idx].replace(" az ", " az-")
            locations_column_names_clean.append(location_column_az_dash)
            locations_clean.append(location_az_dash)     
    
    assert len(locations_clean) == len(locations_column_names_clean), "Something went wrong, the number of elements in the lists are not the same"
    
    return locations_clean, locations_column_names_clean 

In [126]:
def add_counts_of_multiple_location_representations(representation_version, df):
    """
    Sums the counts of multiple representations of the same location into one column of the DataFrame.
    
    Parameters:
    representation_version (str): The pattern to search for in column names.
    df (pandas.DataFrame): The DataFrame to modify.

    Returns:
    pandas.DataFrame: The modified DataFrame with pattern columns added and original columns dropped.
    """
    
    # Identify the columns with the representation version specified
    representation_version_cols = [col for col in df.columns if representation_version in col]
    
    # Iterate over these columns and sum the counts into the first column
    for col in representation_version_cols:
        df[col.replace(representation_version, "")] = df[col.replace(representation_version, "")] + df[col]
        df.drop(col, axis=1, inplace=True)

    return df

#### 1.2.2.1 Country Names

In [127]:
country_names = list(wb_shp.loc[wb_shp["adml"] == 0, "NAME"].values)
country_column_names = list(wb_shp.loc[wb_shp["adml"] == 0, "ID"].values)

#### 1.2.2.2 Province Names

In [128]:
provinces = list(wb_shp.loc[wb_shp["adml"] == 1, "NAME"].values)
provinces_column_names = list(wb_shp.loc[wb_shp["adml"] == 1, "ID"].values)

In [129]:
provinces_clean, provinces_column_names_clean = create_multiple_representation_for_locations(provinces, provinces_column_names)

az article: dayr az zawr


In [130]:
# The CountVectorizer requires as input the numbers of ngrams to consider, so we need to find the maximum number of ngrams required
ngrams_upper_bound_provinces = np.max([len(word.split(" ")) for word in provinces_clean] + [len(word.split("-")) for word in provinces_clean])
print(f"The upper bound for the n-grams is {ngrams_upper_bound_provinces}")

The upper bound for the n-grams is 3


#### 1.2.2.3 District Names

In [131]:
districts = list(wb_shp.loc[wb_shp["adml"] == 2, "NAME"].values)
districts_column_names = list(wb_shp.loc[wb_shp["adml"] == 2, "ID"].values)

In [132]:
districts_clean, districts_column_names_clean = create_multiple_representation_for_locations(districts, districts_column_names)

al article: deir al balah


In [133]:
ngrams_upper_bound_districts = np.max([len(word.split(" ")) for word in districts_clean] + [len(word.split("-")) for word in districts_clean])
print(f"The upper bound for the n-grams is {ngrams_upper_bound_districts}")

The upper bound for the n-grams is 3


#### 1.2.2.4 Count Vectorization and handling duplicates

In [134]:
vocabulary_including_duplicates = category_keywords + country_names + provinces_clean + districts_clean
vocabulary_including_duplicates = [word.lower() for word in vocabulary_including_duplicates]

ngrams_upper_bound = max([ngrams_upper_bound_keywords, ngrams_upper_bound_provinces, ngrams_upper_bound_districts])

column_names_including_duplicates = category_keywords_column_names + country_column_names + provinces_column_names_clean + districts_column_names_clean
column_names_including_duplicates = [name.lower() for name in column_names_including_duplicates]

In [135]:
# Create dictionary with each word and the indexes where it appears (possibly multiple)
duplicate_dict = {}
for element in vocabulary_including_duplicates:
    duplicate_dict[element] = [(word, column_names_including_duplicates[idx], idx) for idx, word in enumerate(vocabulary_including_duplicates) if word == element]
    
# Filter the words and index of the words that appear more than once
duplicates_list = [duplicate for duplicate in duplicate_dict.values() if len(duplicate) > 1]

In [136]:
# Extract the index of the duplicates to remove
print("Duplicates to be removed:")
delete_idx = []
for duplicated_name in duplicates_list:
    for duplicate in duplicated_name[1:]:
        name, column, idx = duplicate
        print(f"word: {name}, column name: {column}")
        delete_idx.append(idx)

# remove the duplicates
vocabulary_without_duplicates = np.delete(np.array(vocabulary_including_duplicates), delete_idx)
column_names_without_duplicates = np.delete(np.array(column_names_including_duplicates), delete_idx)

Duplicates to be removed:
word: food sovereignty, column name: nc_food_sovereignty
word: failed crops, column name: prs_failed_crops
word: basrah, column name: iq_ba_5
word: dahuk, column name: iq_da_2
word: erbil, column name: iq_ar_2
word: kerbala, column name: iq_ka_3
word: kirkuk, column name: iq_ts_4
word: najaf, column name: iq_na_3
word: ajloon, column name: jo_aj_1
word: amman, column name: jo_am_3
word: aqaba, column name: jo_aq_2
word: irbid, column name: jo_ir_9
word: jarash, column name: jo_ja_1
word: karak, column name: jo_ka_7
word: ma'an, column name: jo_mn_5
word: madaba, column name: jo_md_1
word: mafraq, column name: jo_ma_3
word: tafiela, column name: jo_at_3
word: zarqa, column name: jo_az_3
word: beirut, column name: lb_ba_1
word: nabatiye, column name: lb_na_4
word: akkar, column name: lb_ak_1
word: gaza, column name: ps_gz_2
word: ariha, column name: sy_id_1
word: hashimiya, column name: jo_az_2


In [137]:
count = CountVectorizer(vocabulary=vocabulary_without_duplicates, ngram_range=(1, ngrams_upper_bound)).fit_transform(eng["body"].values).toarray()
df = pd.DataFrame(count, columns=column_names_without_duplicates)

In [138]:
# Include the columns for the previously excluded duplicates again
for duplicated_name in duplicates_list:
    _, column_kept, _ = duplicated_name[0]
    for duplicate in duplicated_name[1:]:
        _, column_duplicate, _ = duplicate
        df[column_duplicate] = df[column_kept]
        
# Check if all columns are in the dataframe
assert df.shape[1] == len(column_names_including_duplicates), "The number of columns in the dataframe is not the same as before the cleaning, something went wrong"
assert sum([col not in df.columns for col in column_names_including_duplicates]) == 0, "The column names are not the same as before the cleaning, something went wrong"

In [139]:
# Summing the values of the different columns representing the same location (from the cleaning of the articles and dash) 
df = add_counts_of_multiple_location_representations(representation_version = "_no_dash", df = df)
df = add_counts_of_multiple_location_representations(representation_version = "_al_dash", df = df)
df = add_counts_of_multiple_location_representations(representation_version = "_as_dash", df = df)
df = add_counts_of_multiple_location_representations(representation_version = "_el_dash", df = df)
df = add_counts_of_multiple_location_representations(representation_version = "_ar_dash", df = df)
df = add_counts_of_multiple_location_representations(representation_version = "_az_dash", df = df)
df = add_counts_of_multiple_location_representations(representation_version = "_ath_dash", df = df)

assert df.shape[1] == len(category_keywords) + len(country_names) + len(provinces) + len(districts), "The number of columns in the dataframe is not correct. Cleaning the dash ent did not work. "

In [140]:
df = df[column_order]

#### 1.2.2.5 Merging count data with riginal dataset

In [141]:
eng = eng.merge(df, left_index=True, right_index=True)

In [142]:
eng["kw_all"] = eng[keyword_columns].sum(axis=1)

In [143]:
eng.to_csv(eng_data_folder + "Mashreq_2024-06-23_2024-07-24_daily_article_eng_clean_new.csv", index=False)

## 1.3 Processing Arabic news articles

In [201]:
# Loading the raw downloaded data
ara = pd.read_csv(ara_data_folder + "Mashreq_2024-06-23_2024-07-24_daily_article_ara.csv")

# Adding the body length column
ara["body_len"] = ara["body"].apply(lambda x: len(x))
ara["body_len_str"] = ara["body"].apply(lambda x: len(x.split(" ")))

# Note that Arabic text has only one case, so we don't need to convert it to lowercase

### 1.3.1 Arabic Keywords

In [202]:
keyword_df_ara = keyword_df.loc[~keyword_df["keyword_ara"].isna(), ["keyword_ara", "column_name"]].copy()

In [203]:
keyword_df_ara.drop_duplicates(inplace=True)
keyword_df_ara.reset_index(drop=True, inplace=True)

In [204]:
# Function to add suffix to duplicates
def add_suffix_to_duplicates(column):
    counts = column.value_counts()
    duplicates = counts[counts > 1].index
    counts_dict = {key: 0 for key in duplicates}

    def add_suffix(value):
        if value in counts_dict:
            counts_dict[value] += 1
            return f"{value}_{counts_dict[value]}"
        return value
    return column.apply(add_suffix)

In [205]:
# Apply the function to the 'strings' column
keyword_df_ara['column_name'] = add_suffix_to_duplicates(keyword_df_ara['column_name'])

In [206]:
category_keywords_ara = keyword_df_ara["keyword_ara"].tolist()
category_keywords_column_names_ara = keyword_df_ara["column_name"].tolist()

In [207]:
# The CountVectorizer requires as input the numbers of ngrams to consider, so we need to find the maximum number of ngrams required
ngrams_upper_bound_keywords_ara = np.max([len(word.split(" ")) for word in category_keywords_ara])
print(f"The upper bound for the n-grams is {ngrams_upper_bound_keywords_ara}")

The upper bound for the n-grams is 6


### 1.3.2 Location Names

#### 1.3.2.1 Country Names

In [208]:
country_names_ara = list(wb_shp.loc[wb_shp["adml"] == 0, "NAME_NTVE"].values)
country_column_names_ara = list(wb_shp.loc[wb_shp["adml"] == 0, "ID"].values)

#### 1.3.2.2 Province Names

In [209]:
provinces_ara = list(wb_shp.loc[wb_shp["adml"] == 1, "NAME_NTVE"].values)
provinces_column_names_ara = list(wb_shp.loc[wb_shp["adml"] == 1, "ID"].values)

In [210]:
# The CountVectorizer requires as input the numbers of ngrams to consider, so we need to find the maximum number of ngrams required
ngrams_upper_bound_provinces_ara = np.max([len(word.split(" ")) for word in provinces_ara])
print(f"The upper bound for the n-grams is {ngrams_upper_bound_provinces_ara}")

The upper bound for the n-grams is 3


#### 1.3.2.3 District Names

Note that as of now, we don't have arab district names for Jordan and Palestine, so we don't include them yet.

In [211]:
wb_shp.loc[(wb_shp["adml"] == 2) & (wb_shp["NAME_NTVE"].isna()), "NAME_NTVE"] = ""
wb_shp.loc[wb_shp["adml"] == 2, "NAME_NTVE"] = wb_shp.loc[wb_shp["adml"] == 2, "NAME_NTVE"].apply(lambda x: x.replace("\u200e", ""))

In [212]:
districts_ara = wb_shp.loc[wb_shp["adml"] == 2, "NAME_NTVE"].tolist()
districts_column_names_ara = wb_shp.loc[wb_shp["adml"] == 2, "ID"].tolist()

In [213]:
# The CountVectorizer requires as input the numbers of ngrams to consider, so we need to find the maximum number of ngrams required
ngrams_upper_bound_districts_ara = np.max([len(word.split(" ")) for word in districts_ara])
print(f"The upper bound for the n-grams is {ngrams_upper_bound_districts_ara}")

The upper bound for the n-grams is 3


#### 1.3.2.4 Count Vectorization and handling duplicates

In [214]:
vocabulary_including_duplicates = category_keywords_ara + country_names_ara + provinces_ara + districts_ara
ngrams_upper_bound = max([ngrams_upper_bound_keywords_ara, ngrams_upper_bound_provinces_ara, ngrams_upper_bound_districts_ara])

column_names_including_duplicates = category_keywords_column_names_ara + country_column_names_ara + provinces_column_names_ara + districts_column_names_ara

In [215]:
# Create dictionary with each word and the indexes where it appears (possibly multiple)
duplicate_dict = {}
for element in vocabulary_including_duplicates:
    duplicate_dict[element] = [(word, column_names_including_duplicates[idx], idx) for idx, word in enumerate(vocabulary_including_duplicates) if word == element]
    
# Filter the words and index of the words that appear more than once
duplicates_list = [duplicate for duplicate in duplicate_dict.values() if len(duplicate) > 1]

In [216]:
# Extract the index of the duplicates to remove
print("Duplicates to be removed:")
delete_idx = []
for duplicated_name in duplicates_list:
    for duplicate in duplicated_name[1:]:
        name, column, idx = duplicate
        print(f"word: {name}, column name: {column}")
        delete_idx.append(idx)

# remove the duplicates
vocabulary_without_duplicates = np.delete(np.array(vocabulary_including_duplicates), delete_idx)
column_names_without_duplicates = np.delete(np.array(column_names_including_duplicates), delete_idx)

Duplicates to be removed:
word: طاعون الماشية, column name: pad_cattle_plague
word: هطول الأمطار الضئيل, column name: ws_scarce_rainfall
word: هطول الأمطار الضئيل, column name: ws_scarce_preciptiation_1
word: الحصار, column name: cv_siege
word: السيادة الغذائية, column name: nc_food_sovereignty
word: نقص الأمطار, column name: ws_shortage_of_preciptiation
word: نقص الأمطار, column name: ws_lack_of_rains
word: فشل المحاصيل, column name: nc_failed_crops
word: الانقلاب, column name: pi_coup
word: نفوق الماشية, column name: prs_cattle_death
word: نفوق الماشية, column name: prs_livestock_death
word: المجاعة, column name: nc_famine
word: مرض الحمى القلاعية, column name: nc_foot-and-mouth-disease
word: البصرة, column name: iq_ba_5
word: دهوك, column name: iq_da_2
word: اربيل, column name: iq_ar_2
word: كربلاء, column name: iq_ka_3
word: كركوك, column name: iq_ts_4
word: النجف, column name: iq_na_3
word: السليمانية, column name: iq_sl_10
word: المفرق, column name: jo_ma_3
word: مدينة دمشق, colu

In [217]:
count = CountVectorizer(vocabulary=vocabulary_without_duplicates, ngram_range=(1, ngrams_upper_bound)).fit_transform(ara["body"].values).toarray()
df = pd.DataFrame(count, columns=column_names_without_duplicates)

In [218]:
# Include the columns for the previously excluded duplicates again
for duplicated_name in duplicates_list:
    _, column_kept, _ = duplicated_name[0]
    for duplicate in duplicated_name[1:]:
        _, column_duplicate, _ = duplicate
        df[column_duplicate] = df[column_kept]
        
# Check if all columns are in the dataframe
assert df.shape[1] == len(column_names_including_duplicates), "The number of columns in the dataframe is not the same as before the cleaning, something went wrong"
assert sum([col not in df.columns for col in column_names_including_duplicates]) == 0, "The column names are not the same as before the cleaning, something went wrong"

In [219]:
assert df.shape[1] == len(category_keywords_ara) + len(country_names_ara) + len(provinces_ara) + len(districts_ara), "The number of columns in the dataframe is not correct. Cleaning the dash ent did not work. "

Handling columns of words with multiple arabic translations

In [220]:
# Identify columns with column names containing digits
def has_numbers(inputString):
    return any(char.isdigit() for char in inputString)

duplicate_keywords = [col for col in keyword_df_ara["column_name"].values if has_numbers(col)]
unique_duplicates_without_suffix = np.unique(["_".join(keyword.split("_")[:-1]) for keyword in duplicate_keywords])

# Creating a dictionary with the keys representing the unique english word and the values the multiple representations
duplicate_column_dict = {}
for column in unique_duplicates_without_suffix:
    duplicate_column_dict[column] = [keyword for keyword in duplicate_keywords if column in keyword]
    
# For each key in the dictionary, sum the values of the columns and drop the columns
for column in duplicate_column_dict.keys():
    df[column] = df[duplicate_column_dict[column]].sum(axis=1)
    df.drop(columns=duplicate_column_dict[column], inplace=True)

In [221]:
df = df[column_order]

#### 1.3.2.4 Merging Keywords, Country, provinces and district counts with the original dataset

In [222]:
ara = ara.merge(df, left_index=True, right_index=True)

In [223]:
ara["kw_all"] = ara[keyword_columns].sum(axis=1)

In [224]:
ara.to_csv(ara_data_folder + "Mashreq_2024-06-23_2024-07-24_daily_article_ara_clean.csv", index=False)

# 2. Create summary table

The summary dictionaries count the number of articles that mention at least one keyword of a certain keyword category.
This means that also if multiple keywords of a category are mentioned in an article or an article mentions the same keyword multiple times, 
it only counts as one in the column representing the corresponding keyword category. 

However, an article can be represented with a one in multiple category columns, which is why the sum over all category columns does not correspond to the number of articles mentioning any keyword.
This count is represented by the "kw" column, which is therefore always smaller or equal to the sum of the columns of the different keyword categories. 

If an article mentions multiple locations, it also appears acordingly in multiple rows represented as a one

In [168]:
# Exclude the dates from the language dataframes to make sure that we only include entire weeks
from_date = "2024-06-24"
to_date = "2024-07-21"

In [169]:
eng["date"] = pd.to_datetime(eng["date"])

# English
eng = eng.loc[(eng["date"] >= from_date) & (eng["date"] <= to_date),]

In [170]:
ara["date"] = pd.to_datetime(ara["date"])

# Arabic
ara = ara.loc[(ara["date"] >= from_date) & (ara["date"] <= to_date),]

In [171]:
def get_column_names(df, code, admin_level=-1):
    if admin_level == -1:
        return_list = [col for col in df.columns if col.startswith(code)]
    elif admin_level == 0:
        return_list = [col for col in df.columns if col.startswith(code) and len(col.split("_")) == 1]
    elif admin_level == 1:
        return_list = [col for col in df.columns if col.startswith(code) and len(col.split("_")) == 2]
    elif admin_level == 2:
        return_list = [col for col in df.columns if col.startswith(code) and len(col.split("_")) == 3]
                
    return return_list

In [172]:
def create_summary_df(language_df: pd.DataFrame, country_codes: dict, keyword_codes: dict) -> pd.DataFrame:
    """
    Creates a summary dataframe on a date based on the provided language dataframe. 
    The summary contains the counts of articles from different keyword groups, mentioning province names.
    Articles can be listed multiple times if they mention multiple provinces or keyword groups.

    Args:
        language_df (pd.DataFrame): The language dataframe containing the data.
        country_codes (dict): A dictionary mapping country codes to country names.
        keyword_codes (dict): A dictionary mapping keyword group codes to keyword group names.

    Returns:
        pd.DataFrame: The summary dataframe containing the aggregated information.
    """
    
    # Get all dates from the language dataframe
    unique_dates = language_df["date"].sort_values().unique()

    date_dfs = []
    
    # Iterate over all country codes
    for country_code in tqdm(country_codes.keys()):

        province_columns = get_column_names(language_df, country_code)
        include_country = list(np.repeat(False,len(province_columns))) + list(np.repeat(True,len(province_columns)))
        province_columns = province_columns * 2
        province_columns = [list(item) for item in zip(province_columns, include_country)]
        
        # Iterate over all provinces
        for province_column, include_country in province_columns:
                        
            # Create a dataframe with all unique dates, province names and country names
            date_df = pd.DataFrame(data={"date":unique_dates})
            date_df["location"] = province_column
            date_df["country"] = country_codes[country_code]
            if include_country:
                # Count the number of articles mentioning a certain province for each date, name the columns of this dataframe "date" and "count_articles"
                no_articles = language_df.loc[(language_df[country_code] > 0) & (language_df[province_column] > 0),].groupby("date").size().reset_index()
                no_articles.columns = ["date", "count_articles"]
                
                # Merge the count information with the date dataframe
                date_df = date_df.merge(no_articles, on="date", how="left")
            
                # Iterate over the Keyword Groups
                for keyword_group_code in list(keyword_codes.keys()):

                    # Extract the column names for all columns of the keyword group
                    keyword_group_columns = get_column_names(language_df, keyword_group_code + "_")

                    # Count the number of articles mentioning a certain province and a certain keyword group for each date, name the columns of this dataframe "date" and the keyword group code
                    date_count_df = language_df.loc[(language_df[country_code] > 0) & (language_df[province_column] > 0) & (language_df[keyword_group_columns].sum(axis=1) > 0),].groupby("date")[keyword_group_columns].count().iloc[:,0]
                    date_count_df = pd.DataFrame(date_count_df).reset_index()
                    date_count_df.columns = ["date", keyword_group_code]
                    
                    # Merge the count information with the date dataframe
                    date_df = date_df.merge(date_count_df, on="date", how="left")
                    date_df["include_country"] = include_country
                    
                    for keyword_group_column in keyword_group_columns:
                        date_count_df = language_df.loc[(language_df[country_code] > 0) & (language_df[province_column] > 0) & (language_df[keyword_group_column] > 0),].groupby("date")[keyword_group_column].count()
                        date_count_df = pd.DataFrame(date_count_df).reset_index()
                        date_count_df.columns = ["date", keyword_group_column]
                        
                        # Merge the count information with the date dataframe
                        date_df = date_df.merge(date_count_df, on="date", how="left")

            else:
                # Count the number of articles mentioning a certain province for each date, name the columns of this dataframe "date" and "count_articles"
                no_articles = language_df.loc[(language_df[province_column] > 0),].groupby("date").size().reset_index()
                no_articles.columns = ["date", "count_articles"]
                
                # Merge the count information with the date dataframe
                date_df = date_df.merge(no_articles, on="date", how="left")
                
                # Iterate over the Keyword Groups
                for keyword_group_code in list(keyword_codes.keys()):

                    # Extract the column names for all columns of the keyword group
                    keyword_group_columns = get_column_names(language_df, keyword_group_code + "_")

                    # Count the number of articles mentioning a certain province and a certain keyword group for each date, name the columns of this dataframe "date" and the keyword group code
                    date_count_df = language_df.loc[(language_df[province_column] > 0) & (language_df[keyword_group_columns].sum(axis=1) > 0),].groupby("date")[keyword_group_columns].count().iloc[:,0]
                    date_count_df = pd.DataFrame(date_count_df).reset_index()
                    date_count_df.columns = ["date", keyword_group_code]
                    
                    # Merge the count information with the date dataframe
                    date_df = date_df.merge(date_count_df, on="date", how="left")
                    date_df["include_country"] = include_country
                    
                    for keyword_group_column in keyword_group_columns:
                        date_count_df = language_df.loc[(language_df[province_column] > 0) & (language_df[keyword_group_column] > 0),].groupby("date")[keyword_group_column].count()
                        date_count_df = pd.DataFrame(date_count_df).reset_index()
                        date_count_df.columns = ["date", keyword_group_column]
                        
                        # Merge the count information with the date dataframe
                        date_df = date_df.merge(date_count_df, on="date", how="left")
                   
            date_dfs.append(date_df)
            
    # Concatenate the date dataframes for all the provinces
    summary_df = pd.concat(date_dfs).reset_index(drop=True)
    summary_df["date"] = pd.to_datetime(summary_df["date"])
    summary_df.set_index("date", inplace=True)

    summary_df = summary_df.loc[~((summary_df["location"] == summary_df["country"]) & (summary_df["include_country"] == True))]
    summary_df.drop(columns="kw_all", inplace=True)
    return summary_df

In [173]:
keyword_codes_df = keyword_df.loc[keyword_df["keyword_eng"].isna(), ["column_name", "group_name"]].copy()
keyword_codes = keyword_codes_df.set_index("column_name")["group_name"].to_dict()

In [174]:
country_codes_df = wb_shp.loc[wb_shp["adml"] == 0, ["ID", "NAME"]].copy()
country_codes = country_codes_df.set_index("ID")["NAME"].to_dict()

In [175]:
# English
summary_df_eng = create_summary_df(eng, country_codes, keyword_codes)

100%|██████████| 5/5 [15:14<00:00, 182.84s/it]


In [177]:
summary_df_eng = summary_df_eng.fillna(0)

# Create a new admin level column indicating the level of the administrative division (0: Country, 1: Province, 2: District)
summary_df_eng.insert(2, "admin_level", 0, allow_duplicates=False)
summary_df_eng["admin_level"] = summary_df_eng["location"].str.split("_").apply(lambda x: len(x)) -1

# Excluding columns that searched for combinations of country and (either province or district)
summary_df_eng = summary_df_eng.loc[summary_df_eng["include_country"] == False]
summary_df_eng.drop(columns="include_country", inplace=True)

In [179]:
summary_df_eng.insert(3, "language", "eng", allow_duplicates=False)

In [181]:
summary_df_eng.to_csv(eng_data_folder + "Mashreq_2024-06-23_2024-07-24_daily_article_eng_clean_summary_new.csv")

In [182]:
# Arabic
summary_df_ara = create_summary_df(ara, country_codes, keyword_codes)

100%|██████████| 5/5 [16:14<00:00, 194.93s/it]


In [200]:
summary_df_ara = summary_df_ara.fillna(0)

# Create a new admin level column indicating the level of the administrative division (0: Country, 1: Province, 2: District)
summary_df_ara.insert(2, "admin_level", 0, allow_duplicates=False)
summary_df_ara["admin_level"] = summary_df_ara["location"].str.split("_").apply(lambda x: len(x)) -1

# Excluding columns that searched for combinations of country and (either province or district)
summary_df_ara = summary_df_ara.loc[summary_df_ara["include_country"] == False]
summary_df_ara.drop(columns="include_country", inplace=True)

'summary_df_ara = summary_df_ara.fillna(0)\n\n# Create a new admin level column indicating the level of the administrative division (0: Country, 1: Province, 2: District)\nsummary_df_ara.insert(2, "admin_level", 0, allow_duplicates=False)\nsummary_df_ara["admin_level"] = summary_df_ara["location"].str.split("_").apply(lambda x: len(x)) -1\n\n# Excluding columns that searched for combinations of country and (either province or district)\nsummary_df_ara = summary_df_ara.loc[summary_df_ara["include_country"] == False]\nsummary_df_ara.drop(columns="include_country", inplace=True)'

In [186]:
summary_df_ara.insert(3, "language", "ara", allow_duplicates=False)

In [187]:
summary_df_ara.to_csv(ara_data_folder + "Mashreq_2024-06-23_2024-07-24_daily_article_ara_clean_summary_new.csv")

In [188]:
summary_df = pd.concat([summary_df_eng, summary_df_ara])

In [190]:
summary_df.reset_index(inplace=True)

In [196]:
summary_df.to_csv(data_folder + "/newsapi/summary-dataframes/summary_df_2024_06_23_2024_07_24.csv", index=False)